# LUNAR SOIL COMPOSITION BY TERRAIN


## DATA

Published mean soil compositions from Table 7.15, “Chemical compositions (wt.%) of average soils at lunar landing sites and in selected regions,” in *Lunar Sourcebook: A User's Guide to the Moon* (Heiken, Vaniman, and French, eds., 1991), printed page 346 (PDF page 369). The exact source file is the Lunar and Planetary Institute [`LunarSourceBook.pdf`](https://www.lpi.usra.edu/publications/books/lunar_sourcebook/pdf/LunarSourceBook.pdf).

The repository [data note and extract](../data/lunar_soil_composition/README.md) record the source column for every site and preserve the published wt.% values and precision. All 24 cells were checked against extracted table text and a full-resolution page rendering. Apollo 15 is omitted because the requested terrain grouping excludes its mixed Hadley–Apennine provenance. Oxide labels use Unicode subscripts rather than mathematical notation.


In [ ]:
from pathlib import Path
import numpy as np

repo = Path("..") if Path.cwd().name == "examples" else Path(".")
data_dir = repo / "data/lunar_soil_composition"

data = np.genfromtxt(
    data_dir / "lunar_sourcebook_table_7_15_subset.csv",
    delimiter=",",
    skip_header=1,
    encoding="utf-8",
    dtype=[
        ("terrain", "U10"),
        ("site", "U12"),
        ("source_column", "U3"),
        ("oxide", "U6"),
        ("composition", "f8"),
    ],
)

## PLOT


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lab74

lab74.use()

out = repo / "examples/output"
out.mkdir(exist_ok=True)
oxides = ("TiO₂", "Al₂O₃", "FeO", "MgO")
panels = {
    "Mare sites": ("mare", ("Apollo 11", "Apollo 12", "Apollo 17")),
    "Highlands sites": ("highlands", ("Apollo 14", "Apollo 16", "Luna 20")),
}


def site_values(terrain, site):
    rows = data[(data["terrain"] == terrain) & (data["site"] == site)]
    return np.array(
        [rows["composition"][rows["oxide"] == oxide][0] for oxide in oxides]
    )


fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.75), sharey=True)
group_x = np.arange(len(oxides))
for ax, (title, (terrain, sites)) in zip(axes, panels.items(), strict=True):
    values = np.asarray([site_values(terrain, site) for site in sites])
    lab74.grouped_bar(
        ax,
        values,
        labels=sites,
        categories=oxides,
        positions=group_x,
    )

    ax.set(
        xlim=(-0.55, 3.55),
        ylim=(0, 30),
    )
    ax.yaxis.set_major_locator(mticker.MultipleLocator(5))
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(1))

    lab74.format_frame(ax, style="open")
    lab74.legend(ax, loc="upper left", title=title.upper())

axes[0].set(ylabel="COMPOSITION, WT.%")
fig.subplots_adjust(wspace=0.12)
fig.savefig(out / "05_lunar_soil_composition.png")
plt.close(fig)